In [ ]:
# Updated November 2, 2026. Replace these October 30 sample URIs with your matching run inputs.
# The entrypoint uses the selected MODEL_ID in this checkout; the archive supplies the saved model.
MODEL_ID = "usecase2"

# Prediction consumes the prepared CSV from preprocessing, not the raw GetData CSV.
S3_INPUTS = {
    "prepared_data": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/data/test/prepared/data.csv",
    "model": "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/model/application/model.tar.gz"
}


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import os
import sys
import tarfile

import boto3
import pandas as pd
from botocore.config import Config as ClientConfig
from IPython import get_ipython
from IPython.display import display

# Reload edited model functions too when rerunning the final cell.
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

# Run the Python code in this checkout, not code stored inside the model archive.
REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))
parameter_path = REPOSITORY / "mlops-models/src/mlops_models/models" / MODEL_ID / "parameters.json"

# Rerun setup for fresh inputs; rerun the final cell to debug without downloading again.
work_directory = Path(mkdtemp(prefix="predict-"))
data_directory = work_directory / "prepared"
model_directory = work_directory / "model"
output_directory = work_directory / "output"
output_directory.mkdir()
local_files = {
    "prepared_data": data_directory / "data.csv",
    "model": work_directory / "model.tar.gz",
}

# Set these before importing the entrypoint or the model code.
os.environ.update({
    "MODEL_ID": MODEL_ID,
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_MODEL_ARTIFACT_URI": S3_INPUTS["model"],
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})

# Copy these exact S3 objects unchanged into the local input folders.
s3 = boto3.client("s3", config=ClientConfig(
    connect_timeout=5, read_timeout=15, retries={"total_max_attempts": 2}))
for name, local_file in local_files.items():
    address = urlsplit(S3_INPUTS[name])
    if address.scheme != "s3" or not address.netloc or not address.path.strip("/"):
        raise ValueError(f"Set S3_INPUTS[{name!r}] to an S3 object path in the first cell.")
    local_file.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {S3_INPUTS[name]} -> {local_file}", flush=True)
    s3.download_file(address.netloc, address.path.lstrip("/"), str(local_file))

# model_directory is the extracted model root, not the archive or its parent.
with tarfile.open(local_files["model"]) as archive:
    archive.extractall(model_directory, filter="data")

print("Model directory:", model_directory)
print("Saved model parameters:", model_directory / "parameters.json")
print("Checkout parameters:", parameter_path)
print("Local output directory:", output_directory)


In [ ]:
# Optional checks/inspection; the entrypoint call does not depend on this cell.
for local_file in local_files.values():
    assert local_file.is_file(), f"Missing input: {local_file}"
assert (model_directory / "parameters.json").is_file(), "Expected parameters.json at the extracted model root."
assert parameter_path.is_file(), f"Missing checkout parameters: {parameter_path}"
display(pd.read_csv(data_directory / "data.csv", nrows=10, dtype=str))


In [ ]:
from mlops_models.entrypoints import predict

predict = importlib.reload(predict)
print("Entrypoint:", predict.__file__)
result_directory = predict.execute(
    model_id=MODEL_ID,
    input_path=data_directory,
    output_path=output_directory,
    model_dir=model_directory,
)
print("Outputs:", output_directory)
result_directory
